# Basic Statistics: V-Dem Dataset (v15)

The **Varieties of Democracy (V-Dem)** dataset is one of the largest and most comprehensive datasets on democracy, covering 202 countries from 1789 to 2024. It provides over 4,000 indicators of democracy and governance.

In this notebook, we will:
1. **Load** the dataset
2. **Inspect** its structure (shape, columns, data types)
3. **Select** key democracy indices for analysis
4. **Compute** summary statistics
5. **Visualize** distributions and trends over time

## 1. Setup: Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 120)

## 2. Load the Dataset

In [ ]:
# Load the full V-Dem dataset
df = pd.read_csv("../Data_Raw/VDem/V-Dem-CY-Full+Others-v15.csv")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"Year range: {df['year'].min()} – {df['year'].max()}")
print(f"Number of countries: {df['country_name'].nunique()}")

## 3. Inspect the Dataset Structure

### 3.1 First few rows

In [ ]:
# Preview the first 5 rows (selecting key identifier columns)
df[["country_name", "country_text_id", "country_id", "year", "historical_date"]].head(10)

### 3.2 Data types summary

The dataset contains thousands of columns. Let's summarize the data types.

In [ ]:
# Count of columns by data type
dtype_counts = df.dtypes.value_counts().reset_index()
dtype_counts.columns = ["Data Type", "Number of Columns"]
print("Column data types:\n")
print(dtype_counts.to_string(index=False))

# Missing values overview
total_cells = df.shape[0] * df.shape[1]
missing_cells = df.isna().sum().sum()
print(f"\nTotal cells: {total_cells:,}")
print(f"Missing cells: {missing_cells:,} ({missing_cells/total_cells*100:.1f}%)")

## 4. Select Key Variables for Analysis

According to the V-Dem codebook, the **high-level democracy indices** are the most important summary measures. We focus on:

| Variable | Description |
|---|---|
| `v2x_polyarchy` | **Electoral Democracy Index** – To what extent is the ideal of electoral democracy achieved? |
| `v2x_libdem` | **Liberal Democracy Index** – To what extent is the ideal of liberal democracy achieved? |
| `v2x_partipdem` | **Participatory Democracy Index** – To what extent is the ideal of participatory democracy achieved? |
| `v2x_delibdem` | **Deliberative Democracy Index** – To what extent is the ideal of deliberative democracy achieved? |
| `v2x_egaldem` | **Egalitarian Democracy Index** – To what extent is the ideal of egalitarian democracy achieved? |

All indices range from **0 (lowest)** to **1 (highest)**.

In [ ]:
# Select key variables
id_vars = ["country_name", "country_text_id", "country_id", "year"]
democracy_indices = [
    "v2x_polyarchy",   # Electoral Democracy
    "v2x_libdem",      # Liberal Democracy
    "v2x_partipdem",   # Participatory Democracy
    "v2x_delibdem",    # Deliberative Democracy
    "v2x_egaldem",     # Egalitarian Democracy
]

# Create a focused subset
d = df[id_vars + democracy_indices].copy()
print(f"Subset shape: {d.shape[0]:,} rows × {d.shape[1]:,} columns")
d.head(10)

## 5. Summary Statistics

In [ ]:
# Descriptive statistics for the five democracy indices
summary = d[democracy_indices].describe().round(3)

# Add missing count
summary.loc["missing"] = d[democracy_indices].isna().sum()
summary.loc["non-missing"] = d[democracy_indices].notna().sum()

# Rename columns for readability
pretty_names = {
    "v2x_polyarchy": "Electoral",
    "v2x_libdem": "Liberal",
    "v2x_partipdem": "Participatory",
    "v2x_delibdem": "Deliberative",
    "v2x_egaldem": "Egalitarian",
}
summary.rename(columns=pretty_names)

## 6. Visualization

### 6.1 Distribution of Democracy Indices (All Country-Years)

In [ ]:
# Histogram of all five democracy indices
fig, axes = plt.subplots(1, 5, figsize=(20, 4), sharey=True)

for ax, col in zip(axes, democracy_indices):
    d[col].dropna().hist(bins=40, ax=ax, color="steelblue", edgecolor="white")
    ax.set_title(pretty_names[col], fontsize=13)
    ax.set_xlabel("Index Value")
    ax.set_xlim(0, 1)

axes[0].set_ylabel("Frequency")
fig.suptitle("Distribution of V-Dem Democracy Indices (All Country-Years)", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 6.2 Box Plots of Democracy Indices

In [ ]:
# Box plot comparing all five indices side by side
melted = d.melt(
    id_vars=id_vars,
    value_vars=democracy_indices,
    var_name="Index",
    value_name="Score",
).dropna(subset=["Score"])

melted["Index"] = melted["Index"].map(pretty_names)

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=melted, x="Index", y="Score", palette="Set2", ax=ax)
ax.set_title("Distribution of V-Dem Democracy Indices", fontsize=14)
ax.set_xlabel("")
ax.set_ylabel("Index Score (0–1)")
plt.tight_layout()
plt.show()

### 6.3 Global Trends Over Time

How have democracy levels changed globally? We plot the **global average** of each index by year.

In [ ]:
# Global average of each democracy index by year
yearly_avg = d.groupby("year")[democracy_indices].mean()

fig, ax = plt.subplots(figsize=(12, 5))
for col in democracy_indices:
    ax.plot(yearly_avg.index, yearly_avg[col], label=pretty_names[col], linewidth=1.5)

ax.set_title("Global Average Democracy Indices Over Time", fontsize=14)
ax.set_xlabel("Year")
ax.set_ylabel("Average Index Score")
ax.legend(title="Index", loc="upper left")
ax.set_xlim(yearly_avg.index.min(), yearly_avg.index.max())
ax.set_ylim(0, 0.7)
plt.tight_layout()
plt.show()

### 6.4 Country-Level Comparison (Most Recent Year)

Let's look at the **Electoral Democracy Index** for the most recent year, highlighting the top 20 and bottom 20 countries.

In [ ]:
# Filter to the most recent year
latest_year = d["year"].max()
d_latest = d[d["year"] == latest_year].dropna(subset=["v2x_polyarchy"]).copy()

# Top 20 and bottom 20 countries by Electoral Democracy Index
top20 = d_latest.nlargest(20, "v2x_polyarchy")
bottom20 = d_latest.nsmallest(20, "v2x_polyarchy")

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Top 20
axes[0].barh(
    top20.sort_values("v2x_polyarchy")["country_name"],
    top20.sort_values("v2x_polyarchy")["v2x_polyarchy"],
    color="seagreen",
)
axes[0].set_title(f"Top 20 Countries – Electoral Democracy ({latest_year})", fontsize=13)
axes[0].set_xlabel("Electoral Democracy Index")
axes[0].set_xlim(0, 1)

# Bottom 20
axes[1].barh(
    bottom20.sort_values("v2x_polyarchy", ascending=False)["country_name"],
    bottom20.sort_values("v2x_polyarchy", ascending=False)["v2x_polyarchy"],
    color="indianred",
)
axes[1].set_title(f"Bottom 20 Countries – Electoral Democracy ({latest_year})", fontsize=13)
axes[1].set_xlabel("Electoral Democracy Index")
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.show()

### 6.5 Correlation Between Democracy Indices

How strongly are the five democracy indices correlated with each other?